In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()
using LorentzianSimplexSolver

  Activating project at `~/Documents/Work/effective-spinfoam/code/LorentzianSimplexSolver`
Precompiling project...
   5264.0 ms  ✓ LorentzianSimplexSolver
  1 dependency successfully precompiled in 7 seconds. 168 already precompiled.


In [2]:
# ------------------------------------------------------------
# 1. Precision choice (user-controlled)
# ------------------------------------------------------------
const ScalarT = Float64
#const ScalarT = BigFloat

if ScalarT === BigFloat
    LorentzianSimplexSolver.PrecisionUtils.set_big_precision!(256)
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(sqrt(eps(BigFloat)))
else
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(1e-8)
end

# ------------------------------------------------------------
# 2. Read simplices
# ------------------------------------------------------------
simplices = [[1,2,3,4,6], [1,2,3,5,6], [1,2,4,5,6],[1,2,3,4,7], [1,2,3,5,7], [1,2,4,5,7]]

ns = length(simplices)

all_vertices = unique(Iterators.flatten(simplices))
sort!(all_vertices)

Nverts = length(all_vertices)

# ------------------------------------------------------------
# 3. Read vertex coordinates
# ------------------------------------------------------------
vertex_coords = Dict{Int, Vector{ScalarT}}()    

coords_lines = [
    "0, 0, 0, 0",
    "-0.068000000000000005, -0.21988127663727278, -0.5316227766016838, -1.3316227766016839",
    "0, 0, 0, -3.398088489694245",
    "-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
    "0, 0, -2.942830956382712, -1.6990442448471226",
    "0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
    "-2.4696884592430974, -3.893218630529324, -1.3565336794679874, -1.9090667752920147",
]

for (v, line) in zip(all_vertices, coords_lines)
    vertex_coords[v] = LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
end

In [3]:
# ------------------------------------------------------------
# 4. Build geometry
# ------------------------------------------------------------
datasets = LorentzianSimplexSolver.GeometryTypes.GeometryDataset{ScalarT}[]

for (s, simplex) in enumerate(simplices)
    bdypoints = [vertex_coords[v] for v in simplex]
    # println("Processing simplex $s with vertices $(simplex)...")        
    ds = LorentzianSimplexSolver.GeometryPipeline.run_geometry_pipeline(bdypoints)
    push!(datasets, ds)
end

geom = LorentzianSimplexSolver.GeometryTypes.GeometryCollection(datasets);

In [4]:
# ------------------------------------------------------------
# 6. Connect simplices + face matching + gauge fixing
# ------------------------------------------------------------
if ns > 1
    LorentzianSimplexSolver.KappaOrientation.fix_kappa_signs!(simplices, geom)

    conn = LorentzianSimplexSolver.FourSimplexConnectivity.build_global_connectivity(simplices, geom)
    push!(geom.connectivity, conn)

    LorentzianSimplexSolver.FaceXiMatching.run_face_xi_matching(geom; sector=:ref)
    LorentzianSimplexSolver.GaugeFixingSU.run_su2_su11_gauge_fix(geom);
else
    sl2c = [geom.simplex[i].solgsl2c    for i in 1:ns]
    sgndet = [geom.simplex[i].sgndet    for i in 1:ns]
    geom.simplex[1].solgsl2c = LorentzianSimplexSolver.FaceXiMatching.update_sl2ctest(sl2c, sgndet)[1]
end;

In [5]:
LorentzianSimplexSolver.DefineSymbols.run_define_variables(geom);

In [6]:
sd, _ = LorentzianSimplexSolver.SolveVars.run_solver(geom);

In [7]:
S = LorentzianSimplexSolver.DefineAction.compute_action(geom);

In [8]:
using SymEngine
γ = LorentzianSimplexSolver.DefineAction.γsym()
vals = LorentzianSimplexSolver.ActionEvaluation.build_value_dict(sd, γ; γval=nothing);

In [9]:
S_val = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S, vals);
S_simpl = SymEngine.expand(S_val)

2.40137007368871e-16 + 0.122566574897425*im + (8.5265128291212e-14 - 2.60707391749601*im)*gamma^(-1)

In [10]:
dS = LorentzianSimplexSolver.EOMsHessian.compute_EOMs(S, sd)
dS = LorentzianSimplexSolver.EOMsHessian.check_EOMs(dS, sd; γ=1);

✔ All equations of motion satisfied (γ = 1, tol = 1.0e-8).


In [11]:
Hsym = LorentzianSimplexSolver.EOMsHessian.compute_Hessian(S, sd)
H, labels = LorentzianSimplexSolver.EOMsHessian.evaluate_hessian(Hsym, sd; γ=1);

In [12]:
using LinearAlgebra
eigenvalues = eigvals(H);

In [13]:
vals = sort(eigenvalues, by=abs, rev=true);

In [17]:
vals[end-5:end]

6-element Vector{ComplexF64}:
  -0.0023955376540558838 + 2.0184450421222122e-5im
  -0.0020552774734275575 + 0.0011650285959380756im
  -0.0005276357983400718 - 5.228873774448106e-6im
 -0.00047338999676975935 - 0.00016192647935860702im
 -0.00021685577409013974 + 9.95415246223681e-6im
  -0.0001668582288211058 - 2.262818361676694e-5im

In [15]:
Sregge_num, Sregge_symbol = LorentzianSimplexSolver.ReggeAction.run_Regge_action(geom, γ);

In [16]:
Sregge_symbol

4.44089209850063e-16*j_113*gamma + 9.99200722162641e-16*j_121*gamma + 5.55111512312578e-16*j_132*gamma + 0.361358596793957*j_142*gamma + 0.164783631971831*j_145*gamma + 1.57309531950829*j_151*gamma + 1.37294726304225*j_153*gamma + 7.21644966006352e-16*j_231*gamma + 0.748672769925405*j_241*gamma + 0.361358596793958*j_243*gamma - 0.533724594438381*j_252*gamma + 0.0499182310659404*j_254*gamma - 0.594240703336901*j_342*gamma + 0.905114215737607*j_345*gamma + 3.11347599591839*j_351*gamma - 1.2601519938783*j_353*gamma + 1.11022302462516e-15*j_423*gamma - 1.30837608800203*j_441*gamma - 0.680415275262704*j_443*gamma - 0.680161747863034*j_452*gamma + 0.205852946014386*j_454*gamma + 0.454901334836912*j_542*gamma + 0.0639562559642847*j_545*gamma - 0.786119893920179*j_551*gamma - 1.67116598891946*j_553*gamma - 1.2174765033511*j_641*gamma + 0.377040990895888*j_643*gamma + 1.84136081899391*j_652*gamma + 1.28130376848266*j_654*gamma